# Calling `quantem-cuda` kernels directly

[`quantem-cuda`](https://github.com/electronmicroscopy/quantem-cuda) is an optional companion
package of hand-written CUDA C++ kernels (with analytic gradients) for quantem's compute
hot spots. quantem dispatches to it transparently when it is installed (see the last
section), but every kernel is also a plain public function you can call from any torch
code — they take and return `torch.Tensor`, are registered as torch custom ops, and
compose with autograd and `torch.compile` without graph breaks.

This notebook shows the direct-call API:

1. **Total-variation losses** (`quantem.cuda.core`) — *exactly* what is and isn't
   implemented (squared-anisotropic, isotropic, and per-axis anisotropic-L1), with parity
   checks against pure-torch references.
2. **Using the TV kernels in ptychography workflows** — complex multislice objects,
   per-axis weighting, and what the switch means for your regularizer.
3. **The fused TILTED K-Planes interpolation** (`quantem.cuda.core.ml`).
4. **Transparent dispatch inside quantem** and the kill switch.

**Install** (CUDA 13 toolchain wheels at build time; no system CUDA needed):

```bash
pip install 'nvidia-cuda-nvcc==13.0.*' 'nvidia-cuda-cccl==13.0.*' \
            'nvidia-cuda-runtime==13.0.*' 'nvidia-cuda-crt==13.0.*' 'nvidia-nvvm==13.0.*'
pip install git+https://github.com/electronmicroscopy/quantem-cuda   # PyPI: quantem[cuda], once published
```


In [1]:
import torch

import quantem.cuda
from quantem.cuda.core import tv_loss_iso_3d, tv_loss_sq_3d
from quantem.cuda.core.ml import kplanes_tilted_fuse

print("torch        :", torch.__version__)
print("device       :", torch.cuda.get_device_name(0))
print("quantem-cuda :", quantem.cuda.__version__, "| cudart", quantem.cuda.cudart_version())

torch        : 2.12.0+cu130
device       : NVIDIA RTX PRO 6000 Blackwell Server Edition
quantem-cuda : 0.1.0 | cudart 13000


## 1. The TV-loss kernels — exactly what is implemented

`quantem.cuda.core` ships **three** total-variation functionals, each as a fused forward
kernel plus a fused analytic-backward kernel (one launch each, no large intermediates):

### `tv_loss_sq_3d(volume)` — squared anisotropic TV (quantem `tv_vol` parity)

$$\mathcal{L} = \sum (\Delta_d v)^2 \; + \; \sum (\Delta_h v)^2 \; + \; \sum (\Delta_w v)^2$$

Forward differences along each of the three trailing dims, each squared and summed over
its full index range. **Returns the raw, unnormalized sum** — bit-for-bit the quantity
quantem's tomography `tv_vol` regularizer computes before its `weight / numel` scaling;
you apply your own weighting/normalization.

### `tv_loss_iso_3d(volume, eps=1e-8)` — isotropic (edge-preserving) TV

$$\mathcal{L} = \operatorname{mean}_{\text{corners}} \sqrt{(\Delta_d v)^2 + (\Delta_h v)^2 + (\Delta_w v)^2 + \varepsilon}$$

The three forward differences are evaluated on the common corner set
$[0,D\!-\!2]\times[0,H\!-\!2]\times[0,W\!-\!2]$ and the result is **mean-reduced over
that corner set** (leading dims included). $\varepsilon$ sits *inside* the square root.

### `tv_loss_l1_3d(volume)` — anisotropic L1 TV, per-axis sums

$$\mathcal{L} = \left[\; 	extstyle\sum |\Delta_d v|, \;\; \sum |\Delta_h v|, \;\; \sum |\Delta_w v| \;ight]$$

Forward differences along each trailing dim, absolute values summed per axis over the full
index range, **returned as a shape-`(3,)` tensor of raw, unnormalized per-axis sums**.
Weighting and normalization stay with the caller, so any per-axis scheme is exact —
including ptychography's `(tv_weight_z, tv_weight_xy)` constraint (see section 2). A
size-1 trailing dim contributes an empty (zero) sum. Gradients follow torch's convention
for $|x|$ at zero: $\mathrm{sign}(0) = 0$.

### Shared contract (all kernels)

- input: **fp32, CUDA, real** tensor of shape `[D, H, W]` or `[..., D, H, W]` (`ndim >= 3`);
  any leading batch/channel dims are flattened and included in the reduction;
- returns a 0-dim fp32 tensor on the same device; fully differentiable (hand-written
  backward, gradients exact in one kernel launch); `torch.compile(fullgraph=True)`-safe;
- raises `TypeError`/`ValueError` on non-fp32, non-CUDA, or `ndim < 3` input — there is
  **no CPU fallback** in the package (quantem's dispatch layer handles the fallback).

### Explicitly **not** implemented (so you don't discover it the hard way)

- **Per-axis weights inside a kernel** — `tv_loss_sq_3d` and `tv_loss_iso_3d` weight all
  three axes equally; `tv_loss_l1_3d` solves this by returning the per-axis sums instead
  (and a two-call recipe below recovers per-axis weighting for the squared variant).
- A 2-D-only variant. The anisotropic kernels on `[1, H, W]` degrade gracefully (the
  depth term is an empty sum, so you get pure-2D TV); `tv_loss_iso_3d` on `[1, H, W]` has
  an **empty corner set and returns 0.0** — it needs `D >= 2` to measure anything.
- Complex tensors (take `.angle()` / `.abs()` first), fp16 / bf16 / fp64, Huber/smoothed-L1
  variants, and any normalization other than stated above.

In [2]:
# Parity check: tv_loss_sq_3d == quantem's tv_vol functional (raw sum)
def torch_tv_sq(v):
    return (
        (v[..., 1:, :, :] - v[..., :-1, :, :]).pow(2).sum()
        + (v[..., :, 1:, :] - v[..., :, :-1, :]).pow(2).sum()
        + (v[..., :, :, 1:] - v[..., :, :, :-1]).pow(2).sum()
    )

vol = torch.rand(64, 128, 128, device="cuda", requires_grad=True)
ref, fused = torch_tv_sq(vol), tv_loss_sq_3d(vol)

(g_ref,) = torch.autograd.grad(ref, vol, retain_graph=True)
(g_fused,) = torch.autograd.grad(fused, vol)

print(f"value : torch {ref.item():.4f}  kernel {fused.item():.4f}")
print("grads :", torch.allclose(g_ref, g_fused, rtol=1e-4, atol=1e-5))

value : torch 518014.8750  kernel 518015.3125
grads : True


In [3]:
# Parity check: tv_loss_iso_3d == corner-restricted isotropic TV (mean-reduced)
def torch_tv_iso(v, eps=1e-8):
    # all three forward differences anchored at corner (i, j, k)
    dd = (v[..., 1:, :, :] - v[..., :-1, :, :])[..., :, :-1, :-1]
    dh = (v[..., :, 1:, :] - v[..., :, :-1, :])[..., :-1, :, :-1]
    dw = (v[..., :, :, 1:] - v[..., :, :, :-1])[..., :-1, :-1, :]
    return (dd.pow(2) + dh.pow(2) + dw.pow(2) + eps).sqrt().mean()

ref, fused = torch_tv_iso(vol), tv_loss_iso_3d(vol)
(g_ref,) = torch.autograd.grad(ref, vol, retain_graph=True)
(g_fused,) = torch.autograd.grad(fused, vol)

print(f"value : torch {ref.item():.6f}  kernel {fused.item():.6f}")
print(f"grads : {torch.allclose(g_ref, g_fused, rtol=1e-4, atol=1e-6)} (max abs diff {(g_ref - g_fused).abs().max().item():.1e})")
print("iso on a single slice (D=1) is identically zero:", tv_loss_iso_3d(torch.rand(1, 256, 256, device='cuda')).item())

value : torch 0.653949  kernel 0.653948
grads : True (max abs diff 1.4e-12)
iso on a single slice (D=1) is identically zero: 0.0


## 2. Using the TV kernels in ptychography workflows

quantem's ptychography object constraint (`ObjectConstraints.get_tv_loss`) applies
**anisotropic L1** TV — `mean(|diff|)` per axis, weighted by `(tv_weight_z, tv_weight_xy)`,
averaged over the active axes — to the **phase** of the object (and to the amplitude too
for `obj_type="complex"`).

**Same regularizer, faster: automatic.** `ObjectConstraints._calc_tv_loss` dispatches to
`tv_loss_l1_3d` for fp32 CUDA 3-D arrays (same kill switch as every other dispatch
point), so `tv_weight_z` / `tv_weight_xy` constraints run through the fused kernel with
identical math and gradients — nothing to change in reconstruction code. For direct
calls, the exact composition is verified below: the per-axis raw sums from the kernel,
divided by per-axis difference counts, weighted, and averaged over active axes.

**Different TV flavors, also fast.** The squared and isotropic kernels apply to the same
tensors if you want their behavior instead (quadratic smoothing / classic edge-preserving
isotropic TV):

- the object is complex `(num_slices, H, W)` — call any kernel on `obj.angle()`
  (fp32, real), exactly the tensor the L1 path regularizes (`tv_loss_iso_3d` needs
  `num_slices >= 2`);
- per-axis weighting for the squared variant via two calls (verified below):
  `tv_xy = tv_loss_sq_3d(obj_phase.unsqueeze(1))` (each slice as its own depth-1 volume →
  xy terms only) and `tv_z = tv_loss_sq_3d(obj_phase) - tv_xy`.

In [4]:
# tv_loss_l1_3d: per-axis sums compose to exactly ptychography's _calc_tv_loss
from quantem.cuda.core import tv_loss_l1_3d

def calc_tv_loss_torch(array, weight):  # ObjectConstraints._calc_tv_loss, verbatim
    loss, calc_dim = array.new_zeros(()), 0
    for dim in range(array.ndim):
        w = weight[0] if (dim == 0 and array.ndim == 3) else weight[1]
        if w > 0:
            calc_dim += 1
            loss = loss + w * torch.mean(torch.abs(array.diff(dim=dim)))
    return loss / max(calc_dim, 1)

S, H, W = 16, 512, 512
phase = torch.rand(S, H, W, device="cuda", requires_grad=True)
w_z, w_xy = 5.0, 1e-4

sums = tv_loss_l1_3d(phase)                       # (3,): [Σ|Δz|, Σ|Δy|, Σ|Δx|], raw
counts = sums.new_tensor([(S - 1) * H * W, S * (H - 1) * W, S * H * (W - 1)])
wts = sums.new_tensor([w_z, w_xy, w_xy])
loss_kernel = (wts * sums / counts).sum() / (wts > 0).sum()
loss_torch = calc_tv_loss_torch(phase, (w_z, w_xy))

(g_k,) = torch.autograd.grad(loss_kernel, phase, retain_graph=True)
(g_t,) = torch.autograd.grad(loss_torch, phase)
print(f"loss : kernel {loss_kernel.item():.8f}  torch {loss_torch.item():.8f}")
print("grads:", torch.allclose(g_k, g_t, rtol=1e-4, atol=1e-9))

loss : kernel 0.55573213  torch 0.55573404
grads: True


In [5]:
# Ptychography-shaped demo: complex multislice object, per-axis weighted squared TV
num_slices, H, W = 16, 512, 512
obj = torch.polar(
    torch.rand(num_slices, H, W, device="cuda") + 0.5,
    torch.rand(num_slices, H, W, device="cuda"),
).requires_grad_(True)  # complex64, like a multislice ptycho object

phase = obj.angle()  # fp32 real — same tensor the L1 constraint regularizes

w_z, w_xy = 5.0, 0.1  # e.g. the multislice tutorial's tv_weight_z-dominant setup
tv_xy = tv_loss_sq_3d(phase.unsqueeze(1))      # (S, 1, H, W): depth diffs are empty -> xy only
tv_z = tv_loss_sq_3d(phase) - tv_xy            # all-axis sum minus xy = z only
loss = (w_z * tv_z + w_xy * tv_xy) / phase.numel()
loss.backward()  # flows through .angle() back to the complex object

ref_z = (phase[1:] - phase[:-1]).pow(2).sum()
ref_xy = (phase[:, 1:, :] - phase[:, :-1, :]).pow(2).sum() + (phase[:, :, 1:] - phase[:, :, :-1]).pow(2).sum()
print("per-axis recipe matches torch:",
      torch.allclose(tv_z.detach(), ref_z, rtol=1e-3),
      torch.allclose(tv_xy.detach(), ref_xy, rtol=1e-4))
print("complex-object grad present  :", obj.grad is not None and bool(obj.grad.abs().sum() > 0))

per-axis recipe matches torch: True True
complex-object grad present  : True


In [6]:
# Op-level timings at ptychography- and tomography-representative sizes.
# 'L1' is ptychography's functional (torch chain vs the fused per-axis-sum kernel);
# sq/iso are the other two implemented functionals. fwd+bwd, median of 30.
def _time(fn, x, iters=30):
    for _ in range(5):
        y = fn(x); y.backward(); x.grad = None
    s, e = torch.cuda.Event(True), torch.cuda.Event(True)
    ts = []
    for _ in range(iters):
        torch.cuda.synchronize(); s.record()
        y = fn(x); y.backward(); x.grad = None
        e.record(); torch.cuda.synchronize()
        ts.append(s.elapsed_time(e))
    return sorted(ts)[len(ts) // 2]

def torch_tv_l1(v):  # ptycho _calc_tv_loss, all axis weights equal
    return sum(torch.mean(torch.abs(v.diff(dim=d))) for d in range(v.ndim)) / v.ndim

def kernel_tv_l1(v):
    sums = tv_loss_l1_3d(v)
    d, h, w = v.shape[-3:]
    n = v.numel() // (d * h * w)
    counts = sums.new_tensor([n * (d - 1) * h * w, n * d * (h - 1) * w, n * d * h * (w - 1)]).clamp(min=1)
    return (sums / counts).sum() / 3

print(f"{'shape':>17} | {'L1 torch':>9} {'L1 kernel':>9} | {'sq torch':>9} {'sq kernel':>9} | {'iso torch':>9} {'iso kernel':>10}")
for shape in [(1, 1024, 1024), (16, 512, 512), (16, 1024, 1024), (256, 256, 256)]:
    x = torch.rand(*shape, device="cuda", requires_grad=True)
    row = [_time(torch_tv_l1, x), _time(kernel_tv_l1, x), _time(torch_tv_sq, x), _time(tv_loss_sq_3d, x)]
    if shape[0] > 1:
        row += [_time(torch_tv_iso, x), _time(tv_loss_iso_3d, x)]
        print(f"{str(shape):>17} | {row[0]:>7.2f}ms {row[1]:>7.2f}ms | {row[2]:>7.2f}ms {row[3]:>7.2f}ms | {row[4]:>7.2f}ms {row[5]:>8.2f}ms")
    else:
        print(f"{str(shape):>17} | {row[0]:>7.2f}ms {row[1]:>7.2f}ms | {row[2]:>7.2f}ms {row[3]:>7.2f}ms | {'(D=1: iso n/a)':>20}")

            shape |  L1 torch L1 kernel |  sq torch sq kernel | iso torch iso kernel
  (1, 1024, 1024) |    0.27ms    0.23ms |    0.25ms    0.13ms |       (D=1: iso n/a)
   (16, 512, 512) |    0.34ms    0.26ms |    0.33ms    0.13ms |    0.46ms     0.15ms


 (16, 1024, 1024) |    2.18ms    0.52ms |    1.96ms    0.23ms |    3.23ms     0.36ms


  (256, 256, 256) |    2.20ms    0.52ms |    1.99ms    0.23ms |    3.36ms     0.37ms


**Is it tangible for a ptychography reconstruction?** The TV term runs once per iteration,
so compare the saving against your iteration time. On this GPU (RTX PRO 6000), the fused
L1 kernel — the same functional ptychography uses — is ~4× the torch chain at
`(16, 1024, 1024)` multislice and 256³ sizes (a few ms per iteration) and neutral at
single-slice sizes where launch overhead dominates. Against an FFT-dominated ptychography
iteration (typically tens to hundreds of ms) that is a few-percent end-to-end win at
large multislice sizes and roughly nothing for small single-slice objects — the kernels
exist because *tomography* volumes (256³–512³) needed them. Since the L1 dispatch is
automatic and exact, there is no reason to avoid it; switching functionals (squared /
isotropic) remains a modeling choice, not a performance one.

## 3. The fused TILTED K-Planes interpolation (`quantem.cuda.core.ml`)

`kplanes_tilted_fuse(pts, rotations, plane)` fuses one multiscale level of quantem's
`interpolate_ms_features_tilted` (rotate → tri-plane bilinear sample → Hadamard product)
into a single kernel pair:

- `pts` — fp32 `[B, 3]`, coordinates in $[-1, 1]^3$ (border-clamped outside);
- `rotations` — fp32 `[T, 3, 3]` rotation matrices (the TILTED basis set);
- `plane` — fp32 `[3T, C, H, W]` plane grids, plane index `t*3 + {XY, ZX, YZ}`;
- returns `[B, T*C]` features; gradients flow to **all three** inputs (points → pose,
  rotations → SO(3) parameters, grids → the model).

Sampling semantics match `F.grid_sample(align_corners=True, padding_mode="border")`
exactly, gradients included. It lives in `core.ml` (not a per-technique module) because
`KPlanesTILTED` itself lives in `quantem.core.ml.models.kplanes` — any K-Planes-based
model (tomography object models today, other tensor-decomposition applications tomorrow)
goes through the same op.

In [7]:
from quantem.core import config
from quantem.core.ml.models.kplanes import interpolate_ms_features_tilted
from torch import nn

B, T, C, scales = 200_000, 4, 8, (64, 128)
pts = (torch.rand(B, 3, device="cuda") * 2 - 1).requires_grad_(True)
rot = torch.linalg.qr(torch.randn(T, 3, 3, device="cuda")).Q.contiguous().requires_grad_(True)
grids = nn.ParameterList(
    nn.Parameter(torch.rand(3 * T, C, s, s, device="cuda") * 0.4 + 0.1) for s in scales
)

# direct kernel call, one level at a time (this is all the fused op is):
feats_kernel = torch.cat([kplanes_tilted_fuse(pts, rot, g) for g in grids], dim=-1)

# pure-torch reference path (kill switch forces it):
config.set({"use_cuda_kernels": False})
try:
    feats_torch = interpolate_ms_features_tilted(pts, grids, rot)
finally:
    config.set({"use_cuda_kernels": True})

diff = (feats_kernel - feats_torch).abs().max().item()
print(f"features match torch chain: {torch.allclose(feats_kernel, feats_torch, rtol=1e-3, atol=1e-4)} (max abs diff {diff:.2e})")

def run_kernel(_):
    return torch.cat([kplanes_tilted_fuse(pts, rot, g) for g in grids], dim=-1).square().sum()
def run_torch(_):
    return interpolate_ms_features_tilted(pts, grids, rot).square().sum()

config.set({"use_cuda_kernels": False})
try:
    t_torch = _time(run_torch, pts)
finally:
    config.set({"use_cuda_kernels": True})
t_kernel = _time(run_kernel, pts)
print(f"fwd+bwd, B={B:,}, T={T}, C={C}, scales={scales}: torch {t_torch:.2f} ms | kernel {t_kernel:.2f} ms | {t_torch / t_kernel:.1f}x")

features match torch chain: True (max abs diff 1.76e-06)


fwd+bwd, B=200,000, T=4, C=8, scales=(64, 128): torch 4.27 ms | kernel 1.03 ms | 4.1x


## 4. Transparent dispatch inside quantem

With `quantem-cuda` installed you normally never call the kernels yourself:

- `config.get("has_quantem_cuda")` reports whether the package imported successfully;
- `interpolate_ms_features_tilted` (every `KPlanesTILTED` forward — data term, TV soft
  constraints, volume decoding) dispatches per multiscale level;
- `quantem.tomography.utils.tv_loss_vol_sq` (the tomography `ObjectPixelated` TV
  regularizer) dispatches to `tv_loss_sq_3d`;
- ptychography's `ObjectConstraints._calc_tv_loss` (the `tv_weight_z` / `tv_weight_xy`
  object constraint) dispatches to `tv_loss_l1_3d` — same anisotropic-L1 functional,
  weights, and active-axis normalization;
- dispatch engages only for fp32 CUDA tensors and falls back to the identical pure-torch
  code otherwise;
- `config.set({"use_cuda_kernels": False})` is the global kill switch (default `True`).

Direct calls (as above) remain supported everywhere — dispatch is a convenience, not a
requirement.

In [8]:
print("has_quantem_cuda :", config.get("has_quantem_cuda"))
print("use_cuda_kernels :", config.get("use_cuda_kernels", default=True))

has_quantem_cuda : True
use_cuda_kernels : True
